In [0]:
%run ./01_config

In [0]:
"""
11_backlog_prioritizer.py  —  Decision layer: backlog prioritizer (RQ3)

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 11 — Decision layer: backlog prioritizer (FR4 / RQ3)

# Trains the escalation-probability model of Section 4.3.2 on the ground-truth linkage,
# converts it to an expected cost-of-delay score — probability × consequence, deliberately
# interpretable — and compares the resulting backlog ordering against the priority-code and
# FIFO orderings on the held-out window.

# What this notebook establishes is *ranking quality*: discrimination (AUC) and how much
# realised escalation cost each ordering front-loads. The pre-registered T5 verdict (≥20%
# cost-of-delay reduction) requires the capacity-constrained replay of notebook 12, where
# ordering actually changes outcomes rather than just sequence.

# Models are scikit-learn only (preinstalled): a logistic regression as the primary —
# auditable coefficients, per Section 4.3.2 — and a histogram gradient-boosting comparator.

# Shared configuration from '01_config' is assumed to be in scope.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

use_project_schema()

INK, MUTED, GRID = "#1c1c1c", "#8a8a8a", "#e0e0e0"
ACCENT, WARM, GREEN, PURPLE = "#2b6cb0", "#c05621", "#2f855a", "#6b46c1"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
FIG = f"{FIGURES}/{DATASET_VERSION}_"

def emit(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}{name}.png", bbox_inches="tight")
    fig.savefig(f"{FIG}{name}.svg", bbox_inches="tight", format="svg")
    print(f"saved {FIG}{name}.png / .svg")
    plt.show(); plt.close(fig)

# Load the escalation training set and class cost premiums

if not spark.catalog.tableExists(tbl("gold_escalation_training")):
    raise RuntimeError("gold_escalation_training missing — requires v2.1+ data with the "
                       "escalation linkage; run notebooks 03–06 first.")

esc = spark.table(tbl("gold_escalation_training")).toPandas()
esc["notification_date"] = pd.to_datetime(esc.notification_date)

# consequence: (breakdown settlement + mean downtime) - planned repair cost, per class
prem = spark.sql(f"""
SELECT e.equipment_class,
       PERCENTILE_APPROX(CASE WHEN o.cost_type='BREAKDOWN' THEN o.total_actual_cost END, 0.5)
         + AVG(CASE WHEN o.cost_type='BREAKDOWN' THEN o.downtime_valuation END)
         - PERCENTILE_APPROX(CASE WHEN o.cost_type='PLANNED_REPAIR' THEN o.total_actual_cost END, 0.5)
         AS cost_premium
FROM {tbl('silver_order')} o
JOIN {tbl('silver_equipment')} e ON o.equipment_id = e.equipment_id
GROUP BY e.equipment_class
""").toPandas().set_index("equipment_class")["cost_premium"]

tr, te = esc[esc.split == "train"].copy(), esc[esc.split == "test"].copy()
print(f"defects: train {len(tr):,} ({tr.escalated.mean():.1%} escalated) | "
      f"test {len(te):,} ({te.escalated.mean():.1%} escalated)")

# Features and models

# All features are knowable at notification time — nothing leaks from the outcome window.
# The existing priority code enters as a feature rather than dictating the order, so the
# model subsumes current practice instead of ignoring it.

def design(df, cols=None):
    X = pd.get_dummies(df[["damage_code", "equipment_class"]], drop_first=True).astype(float)
    X["priority"] = df.priority.astype(float)
    X["criticality"] = df.criticality.fillna(2.5).astype(float)
    X["age_years"] = df.equipment_age_days.astype(float) / 365.25
    X["prior_failures"] = df.prior_failures.astype(float)
    if cols is not None:
        X = X.reindex(columns=cols, fill_value=0.0)
    return X

Xtr = design(tr); Xte = design(te, Xtr.columns)
ytr, yte = tr.escalated.values, te.escalated.values

logit = LogisticRegression(max_iter=4000, C=1.0).fit(Xtr, ytr)
hgb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.08,
                                     max_iter=300, random_state=42).fit(Xtr, ytr)

p_logit = logit.predict_proba(Xte)[:, 1]
p_hgb = hgb.predict_proba(Xte)[:, 1]
auc_logit, auc_hgb = roc_auc_score(yte, p_logit), roc_auc_score(yte, p_hgb)
print(f"hold-out AUC:  logistic {auc_logit:.3f}   gradient boosting {auc_hgb:.3f}")

coefs = (pd.Series(logit.coef_[0], index=Xtr.columns)
         .sort_values(key=np.abs, ascending=False).head(12).round(3))
print("\nlargest logistic coefficients (auditable ranking drivers):")
print(coefs.to_string())

# Cost-of-delay score and the three orderings

# score = P(escalation) × class cost premium. Realised escalation cost of each defect is
# its premium if it escalated, zero otherwise — the quantity a good ordering front-loads.

te = te.assign(
    p_escalate=p_logit,
    premium=te.equipment_class.map(prem).fillna(float(prem.median())),
)
te["cod_score"] = te.p_escalate * te.premium
te["realised_cost"] = np.where(te.escalated == 1, te.premium, 0.0)

orderings = {
    "Cost of delay (model)": te.sort_values("cod_score", ascending=False),
    "Priority code": te.sort_values(["priority", "notification_date"]),
    "FIFO": te.sort_values("notification_date"),
}

total_cost = te.realised_cost.sum()
frac = np.arange(1, len(te) + 1) / len(te)
capture = {}
for name, d in orderings.items():
    capture[name] = d.realised_cost.cumsum().values / total_cost

summary = []
for name in orderings:
    c25 = float(np.interp(0.25, frac, capture[name]))
    c50 = float(np.interp(0.50, frac, capture[name]))
    summary.append({"ordering": name, "capture_at_25pct": round(c25, 3),
                    "capture_at_50pct": round(c50, 3)})
summ = pd.DataFrame(summary)
display(spark.createDataFrame(summ))

# Figure P1 — realised escalation cost captured vs backlog worked

fig, ax = plt.subplots(figsize=(7.2, 4.4))
for (name, c), col in zip(capture.items(), [WARM, ACCENT, MUTED]):
    ax.plot(frac, c, lw=1.8, color=col, label=name)
ax.plot([0, 1], [0, 1], color=GRID, lw=1, ls="--")
ax.set_xlabel("fraction of backlog worked (in policy order)")
ax.set_ylabel("share of realised escalation cost addressed")
ax.legend(frameon=False, loc="lower right")
ax.set_title(f"hold-out 2024–25   AUC (logistic) = {auc_logit:.3f}",
             fontsize=8, color=MUTED, loc="left", pad=8)
emit(fig, "p1_cost_capture")

# Figure P2 — calibration by score decile

dec = te.assign(decile=pd.qcut(te.p_escalate, 10, labels=False, duplicates="drop"))
cal = dec.groupby("decile").agg(pred=("p_escalate", "mean"),
                                obs=("escalated", "mean"),
                                n=("escalated", "size")).reset_index()
fig, ax = plt.subplots(figsize=(5.6, 4.2))
lim = max(cal.pred.max(), cal.obs.max()) * 1.15
ax.plot([0, lim], [0, lim], color=MUTED, lw=1)
ax.scatter(cal.pred, cal.obs, s=cal.n / cal.n.max() * 220 + 20,
           color=WARM, edgecolor=INK, lw=.5)
ax.set_xlabel("mean predicted escalation probability (decile)")
ax.set_ylabel("observed escalation rate")
emit(fig, "p2_calibration")

# Example ranked snapshot — Section 4.3.2's contrast table

snap = (te.sort_values("cod_score", ascending=False)
          .assign(priority_rank=lambda d: d.priority.rank(method="first").astype(int))
          .head(10)[["notification_id", "equipment_class", "priority", "criticality",
                     "damage_code", "p_escalate", "premium", "cod_score", "escalated"]]
          .round({"p_escalate": 3, "premium": 0, "cod_score": 0}))
display(spark.createDataFrame(snap.astype(object).where(pd.notna(snap), None)))
print("Note how many top cost-of-delay items carry priority 3–4: the economically urgent work "
      "the priority code demotes is exactly where the ordering benefit lives.")

# Persist

from pyspark.sql.functions import lit

res = pd.DataFrame([
    {"metric": "auc_logistic", "value": round(auc_logit, 4)},
    {"metric": "auc_gradient_boosting", "value": round(auc_hgb, 4)},
    {"metric": "capture_at_25pct_cost_of_delay", "value": summ.loc[0, "capture_at_25pct"]},
    {"metric": "capture_at_25pct_priority", "value": summ.loc[1, "capture_at_25pct"]},
    {"metric": "capture_at_25pct_fifo", "value": summ.loc[2, "capture_at_25pct"]},
    {"metric": "test_defects", "value": len(te)},
    {"metric": "test_escalation_rate", "value": round(float(yte.mean()), 4)},
])
(spark.createDataFrame(res).withColumn("dataset_version", lit(DATASET_VERSION))
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(tbl("results_prioritizer")))
summ.to_csv(f"{EXPORTS}/prioritizer_capture.csv", index=False)
snap.to_csv(f"{EXPORTS}/prioritizer_snapshot.csv", index=False)
cal.to_csv(f"{EXPORTS}/prioritizer_calibration.csv", index=False)
print("results_prioritizer written; exports saved. T5 verdict comes from notebook 12.")